# Part 2 · Waste Classification with CNN
**Owner: Dennis**

### Setup — run first (**Google Colab** or **VS Code**)
**Colab:** download this notebook, open it in Colab, then either (a) upload `realwaste.zip`, `waste_descriptions.csv`, `waste_policy_documents.json` into Colab's **Files** panel, or (b) set `DATASET_URL` below to the `realwaste.zip` GitHub Release link. Turn on a GPU: `Runtime → Change runtime type → T4 GPU`.
**VS Code (with a GPU):** just run — it uses the `data/` folder in the repo.

In [ ]:
import os, sys, zipfile, pathlib, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # optional: realwaste.zip GitHub Release URL (Colab)

if IN_COLAB:
    # if you uploaded a zip to the Files panel, move it into data/
    pathlib.Path("data").mkdir(exist_ok=True)
    for z in ["realwaste.zip"]:
        if os.path.exists(z) and not os.path.exists(f"data/{z}"): shutil.move(z, f"data/{z}")
    for f in ["waste_descriptions.csv","waste_policy_documents.json"]:
        if os.path.exists(f) and not os.path.exists(f"data/{f}"): shutil.move(f, f"data/{f}")
    if DATASET_URL and not os.path.exists("data/realwaste.zip"):
        subprocess.run(["wget","-q","-O","data/realwaste.zip",DATASET_URL], check=False)
    subprocess.run(["pip","-q","install","tensorflow","scikit-learn","seaborn"], check=False)

if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z: z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p),"data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)
IMAGE_DIR = "data/RealWaste"
print("In Colab:", IN_COLAB, "| Images present:", os.path.isdir(IMAGE_DIR))

In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

### 2.1 Build the image datasets (self-contained)

In [ ]:
import tensorflow as tf
data_dir = pathlib.Path(IMAGE_DIR)
class_names = sorted([i.name for i in data_dir.glob("*") if i.is_dir()])
num_classes = len(class_names); print("Classes:", class_names)
train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2,
    subset="training", seed=42, image_size=(IMG_HEIGHT,IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)
validation_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2,
    subset="validation", seed=42, image_size=(IMG_HEIGHT,IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches//2); validation_ds = validation_ds.skip(val_batches//2)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(AUTOTUNE)

### 2.2 Transfer-learning model (MobileNetV2)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
data_aug = models.Sequential([layers.RandomFlip("horizontal"),
                              layers.RandomRotation(0.1), layers.RandomZoom(0.1)])
base = tf.keras.applications.MobileNetV2(input_shape=(IMG_HEIGHT,IMG_WIDTH,3),
                                         include_top=False, weights="imagenet")
base.trainable = False
inp = tf.keras.Input((IMG_HEIGHT,IMG_WIDTH,3))
x = data_aug(inp)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x); x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.3)(x)
out = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inp, out)
model.compile(tf.keras.optimizers.Adam(1e-3), "categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
y = np.concatenate([y.numpy().argmax(1) for _,y in train_ds])
cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(num_classes), y=y)))
cw

### 2.3 Train + fine-tune

In [ ]:
cbs=[tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True),
     tf.keras.callbacks.ReduceLROnPlateau(patience=2,factor=0.3)]
h1 = model.fit(train_ds, validation_data=validation_ds, epochs=12, class_weight=cw, callbacks=cbs)
base.trainable=True
for l in base.layers[:-30]: l.trainable=False
model.compile(tf.keras.optimizers.Adam(1e-5),"categorical_crossentropy",metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=validation_ds, epochs=5, class_weight=cw, callbacks=cbs)

### 2.4 Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
y_true,y_pred=[],[]
for xb,yb in test_dataset:
    pr=model.predict(xb,verbose=0); y_pred+=list(pr.argmax(1)); y_true+=list(yb.numpy().argmax(1))
print("Test accuracy:", accuracy_score(y_true,y_pred))
print(classification_report(y_true,y_pred,target_names=class_names))
sns.heatmap(confusion_matrix(y_true,y_pred),annot=True,fmt="d",xticklabels=class_names,yticklabels=class_names)
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

In [ ]:
def classify_waste_image(image_path):
    """Predicted waste category for an image path (used by Part 5)."""
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_HEIGHT,IMG_WIDTH))
    arr = tf.keras.utils.img_to_array(img)[None,...]
    return class_names[int(model.predict(arr,verbose=0).argmax(1)[0])]

### TODO (Dennis)\n- Justify MobileNetV2 vs EfficientNetB0 (try both)\n- Tune dropout / dense units / fine-tune depth / LR\n- Analyze which categories confuse each other